In [7]:
import os , json
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from datasets import Dataset

In [8]:
SFT_DATA_PATH = r"C:\datasets\llm-data\sft\sft_round1.json"

In [9]:
with open(SFT_DATA_PATH) as f:
    data = json.load(f)

print(len(data))

259


In [10]:
def format_prompt(sample):
    instruction = sample["instruction"].strip()
    input_text  = sample["input"].strip()
    output_text = sample["output"].strip()

    if input_text:
        text = f"""### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}"""
    else:
        text = f"""### Instruction:\n{instruction}\n\n### Response:\n{output_text}"""

    return {"text": text}


In [11]:
formatted = [format_prompt(s) for s in data]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(split['train'])} | Val: {len(split['test'])}")
print(f"\nSample:\n{split['train'][0]['text'][:300]}")

Train: 233 | Val: 26

Sample:
### Instruction:
How does matrix multiplication work in NumPy and what are the computational complexity considerations when multiplying large matrices?

### Response:
### Reasoning:
Matrix multiplication follows the rule that for matrices A(m×n) and B(n×p), the result C(m×p) where C[i,j] = Σ(A[i,k] 


In [21]:
from transformers import AutoTokenizer , AutoModelForCausalLM , TrainingArguments , BitsAndBytesConfig
from peft import LoraConfig , prepare_model_for_kbit_training
from trl import SFTTrainer
import torch , gc

In [22]:
MODEL_PATH  = r"C:\datasets\llm-data\models\phi3-pretrained-merged"
OUTPUT_PATH = r"C:\datasets\llm-data\models\phi3-sft"

In [23]:
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

Free VRAM: 4.96 GB


In [24]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [26]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH , quantization_config = bnb_config , device_map={"": 0} , local_files_only=True)

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\quantizers\auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\utils\modeling.py:416: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  new_value = value.to(device)


In [27]:
model = prepare_model_for_kbit_training(model)

In [28]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False
)

In [29]:
training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
)

In [30]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    tokenizer=tokenizer,
    max_seq_length=512,
    peft_config=lora_config,
    dataset_text_field="text",
)

c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\trl\trainer\sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\trl\trainer\sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/233 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [31]:
trainer.train()

  0%|          | 0/87 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
You are not running the flash-attention implementation, expect numerical differences.
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


{'loss': 0.5199, 'grad_norm': 0.09107968211174011, 'learning_rate': 0.0002, 'epoch': 0.69}
{'loss': 0.4105, 'grad_norm': 0.1271953135728836, 'learning_rate': 0.00014029850746268658, 'epoch': 1.37}


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


  0%|          | 0/26 [00:00<?, ?it/s]

{'eval_runtime': 64.9256, 'eval_samples_per_second': 0.4, 'eval_steps_per_second': 0.4, 'epoch': 1.72}


c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


{'loss': 0.3334, 'grad_norm': 0.15835706889629364, 'learning_rate': 8.059701492537314e-05, 'epoch': 2.06}
{'loss': 0.2986, 'grad_norm': 0.22646231949329376, 'learning_rate': 2.0895522388059702e-05, 'epoch': 2.75}
{'train_runtime': 18970.3344, 'train_samples_per_second': 0.037, 'train_steps_per_second': 0.005, 'train_loss': 0.38213029264033527, 'epoch': 2.99}


TrainOutput(global_step=87, training_loss=0.38213029264033527, metrics={'train_runtime': 18970.3344, 'train_samples_per_second': 0.037, 'train_steps_per_second': 0.005, 'total_flos': 7700614900992000.0, 'train_loss': 0.38213029264033527, 'epoch': 2.9871244635193133})

In [34]:
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

('C:\\datasets\\llm-data\\models\\phi3-sft\\tokenizer_config.json',
 'C:\\datasets\\llm-data\\models\\phi3-sft\\special_tokens_map.json',
 'C:\\datasets\\llm-data\\models\\phi3-sft\\tokenizer.model',
 'C:\\datasets\\llm-data\\models\\phi3-sft\\added_tokens.json',
 'C:\\datasets\\llm-data\\models\\phi3-sft\\tokenizer.json')